# RD-IV main results (Wald ratio)
Compute local IV effects as the ratio of RD reduced-form to RD first-stage.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.paths import PAPER_TABLES_DIR
from src.viz_style import set_style

In [2]:
set_style()

first_stage_path = PAPER_TABLES_DIR / "rd_first_stage.csv"
reduced_form_path = PAPER_TABLES_DIR / "rd_reduced_form.csv"

for path in [first_stage_path, reduced_form_path]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required input: {path}")

first_stage = pd.read_csv(first_stage_path)
reduced_form = pd.read_csv(reduced_form_path)

PRIMARY_HORIZONS = [1, 2, 4]

rows = []
for outcome in ["log_gdp_cum", "inv_share_avg", "inflation_path"]:
    for h in PRIMARY_HORIZONS:
        tau_e = first_stage.loc[
            (first_stage["outcome"] == "efw_summary") & (first_stage["horizon"] == h)
        ]
        tau_y = reduced_form.loc[(reduced_form["outcome"] == outcome) & (reduced_form["horizon"] == h)]
        if tau_e.empty or tau_y.empty:
            continue

        tau_e_val = float(tau_e["coef"].iloc[0])
        tau_e_se = float(tau_e["se"].iloc[0])
        tau_y_val = float(tau_y["coef"].iloc[0])
        tau_y_se = float(tau_y["se"].iloc[0])

        if not np.isfinite(tau_e_val) or tau_e_val == 0:
            continue

        beta = tau_y_val / tau_e_val
        # Delta method (ignores covariance)
        se_beta = np.sqrt((tau_y_se / tau_e_val) ** 2 + (tau_y_val * tau_e_se / (tau_e_val**2)) ** 2)

        rows.append(
            {
                "outcome": outcome,
                "horizon": h,
                "beta": float(beta),
                "se": float(se_beta),
                "tau_e": tau_e_val,
                "tau_y": tau_y_val,
                "n_left": int(tau_e["n_left"].iloc[0]) if "n_left" in tau_e.columns else np.nan,
                "n_right": int(tau_e["n_right"].iloc[0]) if "n_right" in tau_e.columns else np.nan,
            }
        )

iv_df = pd.DataFrame(rows)
PAPER_TABLES_DIR.mkdir(parents=True, exist_ok=True)
iv_path = PAPER_TABLES_DIR / "rd_iv_main.csv"
iv_df.to_csv(iv_path, index=False)

display(iv_df.style.set_caption("RD-IV (Wald) estimates"))

,outcome,horizon,beta,se,tau_e,tau_y,n_left,n_right
0,log_gdp_cum,1,-0.100581,0.107007,-0.150552,0.015143,38,30
1,log_gdp_cum,2,-0.096075,0.143247,-0.172166,0.016541,41,31
2,log_gdp_cum,4,-0.695664,1.078157,-0.062650,0.043583,50,53
3,inv_share_avg,1,-10.540512,10.272735,-0.150552,1.586893,38,30
4,inv_share_avg,2,-7.148705,8.553426,-0.172166,1.230766,41,31
5,inv_share_avg,4,-27.876163,41.939881,-0.062650,1.746441,50,53
6,inflation_path,1,-2.424380,5.079794,-0.150552,0.364995,38,30
7,inflation_path,2,-2.793706,5.105536,-0.172166,0.480982,41,31
8,inflation_path,4,-28.004232,40.650107,-0.062650,1.754465,50,53


## Interpretation
The IV estimates use the Wald ratio of RD reduced-form to first-stage
discontinuities at the cutoff (primary horizons h=1,2,4).